# Road Damage Detection — YOLO Train, Compare, Deploy

Before running: **Runtime > Change runtime type > T4 GPU**. Run cells top to bottom.

In [1]:
!pip install -q -U ultralytics roboflow gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 134.9 MB/s eta 0:00:00


In [2]:
import os
import glob
import time
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from roboflow import Roboflow

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 1. Dataset

In [ ]:
import os
from roboflow import Roboflow
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ROBOFLOW_API_KEY")

if not API_KEY:
    raise ValueError("API Key not found! Make sure your .env file exists and contains ROBOFLOW_API_KEY.")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("pine-bruiser").project("road-damage-l1ju7-8iwva")
version = project.version(1)
dataset = version.download("yolov8")

DATA_YAML = os.path.join(dataset.location, "data.yaml")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to road-damage-1 in yolov8:: 100%|██████████| 13519/13519 [00:16<00:00, 809.56it/s] 


In [6]:
for split_name in ["train", "valid", "test"]:
    imgs = glob.glob(os.path.join(dataset.location, split_name, "images", "*"))
    print(split_name, len(imgs))

train 4930
valid 1146
test 681


## 2. Config

In [7]:
MODEL_LIST = ["yolov8n.pt", "yolo11n.pt", "yolo26n.pt"]
EPOCHS = 25
TUNE_EPOCHS = 5
IMG_SIZE = 640
BATCH = 16
TRAIN_FRACTION = 1.0
EVAL_SPLIT = "valid"
TUNE_CONFIGS = [
    {"lr0": 0.01, "optimizer": "SGD"},
    {"lr0": 0.001, "optimizer": "AdamW"},
]

## 3. Train + light hyperparameter search + evaluate

In [11]:
def tune_and_train(model_name):
    best_cfg = TUNE_CONFIGS[0]
    best_score = -1
    for cfg in TUNE_CONFIGS:
        trial = YOLO(model_name)
        trial.train(
            data=DATA_YAML,
            epochs=TUNE_EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            fraction=TRAIN_FRACTION,
            lr0=cfg["lr0"],
            optimizer=cfg["optimizer"],
            project="tune_runs",
            name=f"{model_name.replace('.pt', '')}_{cfg['optimizer']}",
            plots=False,
            verbose=False,
            exist_ok=True,
        )
        trial_metrics = trial.val(data=DATA_YAML, split=EVAL_SPLIT, verbose=False)
        score = trial_metrics.box.map
        if score > best_score:
            best_score = score
            best_cfg = cfg

    final_model = YOLO(model_name)
    final_model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        lr0=best_cfg["lr0"],
        optimizer=best_cfg["optimizer"],
        project="final_runs",
        name=model_name.replace(".pt", ""),
        plots=True,
        exist_ok=True,
    )
    weights_path = os.path.join("final_runs", model_name.replace(".pt", ""), "weights", "best.pt")
    return weights_path, best_cfg

In [12]:
def measure_fps(weights_path, n_samples=30):
    eval_model = YOLO(weights_path)
    sample_imgs = []
    for split_name in ["valid", "test", "train"]:
        found = glob.glob(os.path.join(dataset.location, split_name, "images", "*"))
        if found:
            sample_imgs = found[:n_samples]
            break
    eval_model.predict(sample_imgs[0], verbose=False)
    start = time.time()
    for img_path in sample_imgs:
        eval_model.predict(img_path, verbose=False)
    elapsed = time.time() - start
    return len(sample_imgs) / elapsed

In [ ]:
DATA_YAML = os.path.join(dataset.location, "data.yaml")
comparison_rows = []

for model_name in MODEL_LIST:
    weights_path, cfg_used = tune_and_train(model_name)
    eval_model = YOLO(weights_path)
    val_metrics = eval_model.val(data=DATA_YAML, split=EVAL_SPLIT, imgsz=IMG_SIZE, verbose=False)
    fps = measure_fps(weights_path)
    size_mb = os.path.getsize(weights_path) / (1024 * 1024)

    comparison_rows.append({
        "model": model_name,
        "lr0": cfg_used["lr0"],
        "optimizer": cfg_used["optimizer"],
        "precision": val_metrics.box.mp,
        "recall": val_metrics.box.mr,
        "mAP50": val_metrics.box.map50,
        "mAP50_95": val_metrics.box.map,
        "fps": fps,
        "size_mb": size_mb,
        "weights_path": weights_path,
    })
    print(model_name, "done")

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road-damage-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_SGD, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_m

## 4. Comparison table and plots

In [ ]:
df = pd.DataFrame(comparison_rows)
df["efficiency_score"] = df["fps"] / df["size_mb"]
df = df.sort_values("mAP50_95", ascending=False).reset_index(drop=True)
df.to_csv("model_comparison.csv", index=False)
df

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].bar(df["model"], df["mAP50_95"])
axes[0].set_title("mAP50-95")
axes[1].bar(df["model"], df["fps"])
axes[1].set_title("Inference FPS")
axes[2].bar(df["model"], df["size_mb"])
axes[2].set_title("Model Size (MB)")
axes[3].bar(df["model"], df["efficiency_score"])
axes[3].set_title("FPS per MB (efficiency)")
plt.tight_layout()
plt.savefig("model_comparison.png")
plt.show()

## 5. Pick the best model

In [ ]:
best_row = df.iloc[0]
shutil.copy(best_row["weights_path"], "best_model.pt")
print("Best model:", best_row["model"])
print(best_row)

## 6. Deployment app (Gradio)

In [ ]:
import gradio as gr
from PIL import Image

deploy_model = YOLO("best_model.pt")

def detect_damage(image, conf):
    results = deploy_model.predict(image, conf=conf, verbose=False)
    annotated = results[0].plot()[..., ::-1]
    names = results[0].names
    counts = {}
    for c in results[0].boxes.cls.tolist():
        label = names[int(c)]
        counts[label] = counts.get(label, 0) + 1
    summary = ", ".join(f"{k}: {v}" for k, v in counts.items()) if counts else "No damage detected"
    return Image.fromarray(annotated), summary

demo = gr.Interface(
    fn=detect_damage,
    inputs=[gr.Image(type="pil", label="Upload Road Image"), gr.Slider(0.1, 0.9, value=0.25, label="Confidence Threshold")],
    outputs=[gr.Image(type="pil", label="Detected Damage"), gr.Textbox(label="Detected Classes")],
    title="Road Damage Detection",
)
demo.launch(share=True)